## 4 — GLMERStan Climate  
Bayesian Multilevel Regression + Poststratification  
Outcome: `happening_bin` (Do you think global warming is happening? Yes=1 / No=0)  
Model: `happening_bin ~ (1|gender) + (1|state_fips) + (1|race4) + (1|educ_category)`  
Estimator: NUTS via bambi / PyMC  (Python equivalent of `rstanarm::stan_glmer`)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["PYTENSOR_FLAGS"] = "floatX=float64"

import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
import matplotlib.pyplot as plt
import time

DATA_DIR = "/Users/carmenk/Documents/GitHub/MRdeeP-Deep-Learning-MRP/test_data/processed/"

# ── reproducibility ───────────────────────────────────────────────────────────
SEED    = 42
CHAINS  = 2
DRAWS   = 1000   # posterior draws per chain
TUNE    = 1000   # warmup draws per chain
# For a quicker test run set DRAWS=500, TUNE=500
TARGET_ACCEPT = 0.9

### 1. Load data

In [ ]:
survey_raw = pd.read_csv(DATA_DIR + "climate_survey_responses_recoded.csv",
                        dtype={"state_fips": str})
ps_frame   = pd.read_csv(DATA_DIR + "poststrat_state.csv",
                        dtype={"state_fips": str})

print(f"Survey raw:   {survey_raw.shape}")
print(f"Poststrat:    {ps_frame.shape} | {ps_frame['state_fips'].nunique()} states")

### 2. Prep survey data  
Drop Don't-Know responses for `happening_bin`, align category types.

In [ ]:
DEMOG = ["gender", "race4", "educ_category", "state_fips"]

# Drop DK (NaN happening_bin)
survey = survey_raw.dropna(subset=["happening_bin"]).copy()
survey["happening_bin"] = survey["happening_bin"].astype(int)

# educ_category as string so bambi treats it as categorical grouping factor
survey["educ_category"] = survey["educ_category"].astype(str)
ps_frame["educ_category"] = ps_frame["educ_category"].astype(str)

print(f"Respondents after dropping DK: {len(survey):,}  "
      f"({survey['happening_bin'].mean()*100:.1f}% Yes)")
print()
print("Level counts in survey:")
for c in DEMOG:
    print(f"  {c}: {sorted(survey[c].unique().tolist())}")

### 3. Fit Bayesian mixed-effects logistic regression  
Formula mirrors the R specification:  
```r
stan_glmer(happening_bin ~ (1|gender) + (1|state_fips) + (1|race4) + (1|educ_category),
           family = binomial(link='logit'), data = dat)
```  
bambi applies weakly-informative Student-t priors on the intercept and half-Normal priors on group SDs — same defaults as rstanarm.

In [ ]:
FORMULA = "happening_bin ~ (1|gender) + (1|state_fips) + (1|race4) + (1|educ_category)"

model = bmb.Model(
    formula  = FORMULA,
    data     = survey[DEMOG + ["happening_bin"]],
    family   = "bernoulli",
)
model.build()
print(model)

In [ ]:
start = time.time()
idata = model.fit(
    draws          = DRAWS,
    tune           = TUNE,
    chains         = CHAINS,
    random_seed    = SEED,
    target_accept  = TARGET_ACCEPT,
    progressbar    = True,
)
elapsed = time.time() - start
print(f"\nSampling complete in {elapsed/60:.1f} min")

### 4. Convergence diagnostics

In [ ]:
# R-hat and ESS for key parameters
diag = az.summary(idata, var_names=["Intercept",
                                     "1|gender_sigma", "1|state_fips_sigma",
                                     "1|race4_sigma",  "1|educ_category_sigma"],
                  round_to=3)
print(diag[["mean","sd","hdi_3%","hdi_97%","r_hat","ess_bulk"]])

In [ ]:
# Trace plots for group-level SDs
az.plot_trace(idata, var_names=["1|gender_sigma","1|state_fips_sigma",
                                 "1|race4_sigma","1|educ_category_sigma"],
              compact=True)
plt.tight_layout()
plt.show()

### 5. Posterior predictive on the poststrat frame  
Equivalent of R's `posterior_epred(mod, newdata=bench, draws=1000)`:  
for each demographic cell in the frame, draw the posterior expected probability of Yes.

In [ ]:
# Predict mean probabilities over posterior draws for every demographic cell
idata_pred = model.predict(
    idata,
    data      = ps_frame[DEMOG],
    kind      = "mean",      # expected value (probability), not 0/1 samples
    inplace   = False,
)

# posterior["happening_bin_mean"] shape: (chains, draws, n_cells)
pred_matrix = idata_pred.posterior["happening_bin_mean"].values
n_cells = pred_matrix.shape[-1]
pred_matrix = pred_matrix.reshape(-1, n_cells)   # (total_draws, n_cells)

# Average across all posterior draws → one probability per cell
ps_frame["predicted_prob"] = pred_matrix.mean(axis=0)

print(f"Predicted probabilities: "
      f"min={ps_frame['predicted_prob'].min():.3f}  "
      f"mean={ps_frame['predicted_prob'].mean():.3f}  "
      f"max={ps_frame['predicted_prob'].max():.3f}")
ps_frame[["state_fips","gender","race4","educ_category",
          "N_rounded","predicted_prob"]].head(8)

### 6. Poststratification  
Weighted average of cell-level predicted probabilities, weighted by ACS population counts.

In [ ]:
result = (
    ps_frame
    .groupby("state_fips")
    .apply(lambda g: np.average(g["predicted_prob"], weights=g["N_rounded"]),
           include_groups=False)
    .reset_index(name="happening_estimate")
)

# Merge state names for readability
STATE_NAMES = {
    '01':'Alabama','02':'Alaska','04':'Arizona','05':'Arkansas','06':'California',
    '08':'Colorado','09':'Connecticut','10':'Delaware','11':'District of Columbia',
    '12':'Florida','13':'Georgia','15':'Hawaii','16':'Idaho','17':'Illinois',
    '18':'Indiana','19':'Iowa','20':'Kansas','21':'Kentucky','22':'Louisiana',
    '23':'Maine','24':'Maryland','25':'Massachusetts','26':'Michigan',
    '27':'Minnesota','28':'Mississippi','29':'Missouri','30':'Montana',
    '31':'Nebraska','32':'Nevada','33':'New Hampshire','34':'New Jersey',
    '35':'New Mexico','36':'New York','37':'North Carolina','38':'North Dakota',
    '39':'Ohio','40':'Oklahoma','41':'Oregon','42':'Pennsylvania',
    '44':'Rhode Island','45':'South Carolina','46':'South Dakota',
    '47':'Tennessee','48':'Texas','49':'Utah','50':'Vermont',
    '51':'Virginia','53':'Washington','54':'West Virginia','55':'Wisconsin',
    '56':'Wyoming',
}
result["state_name"] = result["state_fips"].map(STATE_NAMES)
result = result.sort_values("happening_estimate", ascending=False).reset_index(drop=True)

print(f"National weighted estimate: {np.average(result['happening_estimate']):.3f}")
print()
print("Top 10 states (highest % believing GW is happening):")
print(result.head(10).to_string(index=False))
print()
print("Bottom 10 states:")
print(result.tail(10).to_string(index=False))

### 7. Visualise state estimates

In [ ]:
fig, ax = plt.subplots(figsize=(8, 12))
plot_df = result.sort_values("happening_estimate")
ax.barh(plot_df["state_name"], plot_df["happening_estimate"], color="steelblue", alpha=0.8)
ax.axvline(np.average(result["happening_estimate"]), color="red",
           linestyle="--", linewidth=1.2, label="National avg")
ax.set_xlabel("P(GW is happening)", fontsize=12)
ax.set_title("GLMERStan — State-level MRP Estimates\n'Is global warming happening?'",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### 8. Save results

In [ ]:
OUT = DATA_DIR + "glmerstan_happening_estimates.csv"
result[["state_fips","state_name","happening_estimate"]].to_csv(OUT, index=False)
print(f"Saved → {OUT}")
print(result[["state_fips","state_name","happening_estimate"]].to_string(index=False))